In [ ]:
import time
import requests
import pandas as pd
from pprint import pprint
from requests.auth import HTTPBasicAuth     # Standard HTTP authentication method that sends
                                            #  username and password with each request,
                                            #  encoded in Base64
from google.colab import userdata

In [ ]:
#calling Census ACS API


API_KEY = userdata.get('Census-API')   # name of your Colab secret
Census_url = 'https://api.census.gov/data/2024/acs/acs5'
params_census = {
    'get': 'NAME,B01003_001E,B19013_001E',   # name, total population, median income
    'for': 'place:*',                         # every city/town...
    'in': 'state:*',                          # ...in every state
    'key': API_KEY,
}

try:
    response = requests.get(Census_url, params=params_census, timeout=30)
    print(response.status_code)
    if response.status_code == 200:
        data = response.json()
        pprint(data[0:30])          # header row + first 30 cities
    else:
        print(response.text)      # Census explains what went wrong
except Exception as e:
    print('Error occurred:', e)   # show the real error


In [ ]:
#CitiBike Data by city

citibike_url='https://api.citybik.es/v2/networks/'

try:
    response_citi = requests.get(citibike_url, timeout=30)
    if response_citi.status_code == 200:
        data_citibike = response_citi.json()
        pprint(data_citibike['networks'])          # header row + first 10 cities

    else:
        print(response_citi.text)      # Citibik explains what went wrong

except Exception as e:
    print('Error occurred:', e)   # show the real error


In [ ]:
# flatten the nested "location" dict into columns, then filter to the US
networks_df = pd.json_normalize(data_citibike['networks'])
us_networks = networks_df[networks_df['location.country'] == 'US']
print(len(us_networks), 'US networks')
us_networks[['id', 'name', 'location.city']].head(30)


In [ ]:
citibike_url_nyc='https://api.citybik.es/v2/networks/citi-bike-nyc'

try:
    response_citi_nyc = requests.get(citibike_url_nyc, timeout=30)
    if response_citi_nyc.status_code == 200:
        data_citibike_nyc = response_citi_nyc.json()
        #Layout of the data to be put into a dataframe
        pprint(data_citibike_nyc['network']['stations'][20])          # header row + first 10 cities

    else:
        print(response_citi_nyc.text)      # Citi explains what went wrong

except Exception as e:
    print('Error occurred:', e)   # show the real error


In [ ]:
import time

def fetch_stations(network_id):
    """Fetch all stations for one network; return a list of dicts (or [] on failure)."""
    url = f'https://api.citybik.es/v2/networks/{network_id}'
    try:
        response = requests.get(url, timeout=30)
        if response.status_code == 200:
            stations = response.json()['network']['stations']
            for s in stations:
                s['network_id'] = network_id      # tag each station with its network
            return stations
        print(network_id, 'failed:', response.status_code)
    except Exception as e:
        print(network_id, 'error:', e)
    return []

all_stations = []
for net_id in us_networks['id']:
    all_stations.extend(fetch_stations(net_id))
    time.sleep(1)                                 # be polite to the API


#organizing total bike docs per city adding it to the dataframe
stations_df = pd.json_normalize(all_stations)
stations_df['total_docks'] = stations_df['free_bikes'] + stations_df['empty_slots']
#
summary = (stations_df.groupby('network_id')
           .agg(station_count=('id', 'count'), total_docks=('total_docks', 'sum'))
           .sort_values('station_count', ascending=False))
summary.head(30)